# AutoGluon Time Series In-Depth

원본 튜토리얼: https://auto.gluon.ai/dev/tutorials/timeseries/forecasting-indepth.html

이 노트북은 `TimeSeries_quick_start.ipynb` 다음 단계로, AutoGluon 시계열 예측에서 자주 등장하는 핵심 개념을 한 번에 정리합니다. 

- 확률적 시계열 예측과 `prediction_length`
- `static features`, `known covariates`, `past covariates`
- irregular index와 missing value 처리
- `evaluate()`, `leaderboard()`, backtesting
- 모델 종류와 `TimeSeriesPredictor` 고급 설정

각 섹션은 "이 개념이 왜 필요한가", "AutoGluon은 이를 어떻게 해석하는가", "실제로 어떤 코드로 연결되는가" 순서로 읽으면 이해하기 쉽습니다.


## 1. 라이브러리 불러오기

`pandas`와 `numpy`는 데이터를 읽고 가공하기 위한 기본 도구입니다. `matplotlib`은 예측 구간을 시각적으로 확인할 때 사용합니다.

여기서 정말 중요한 두 클래스는 아래와 같습니다.

- `TimeSeriesDataFrame`: AutoGluon이 시계열 데이터를 이해하기 위한 전용 자료구조입니다. 여러 개의 시계열을 `item_id`와 `timestamp` 기준으로 정리해 둔 형태라고 생각하면 됩니다.
- `TimeSeriesPredictor`: 시계열 문제의 설정을 정의하고, 모델 학습(`fit`), 예측(`predict`), 평가(`evaluate`), 모델 비교(`leaderboard`)를 담당하는 핵심 클래스입니다.

즉, 보통의 흐름은 `DataFrame`을 `TimeSeriesDataFrame`으로 바꾼 뒤, `TimeSeriesPredictor`에 넣어서 학습과 평가를 진행하는 순서입니다.


In [1]:
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


/Users/munju/Documents/project/TIL-github/TIL/2026/RAPIDS/2_AutoML_study/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 확률적 시계열 예측 이해하기

시계열 예측은 과거 관측값을 바탕으로 미래 구간을 예측하는 작업입니다. 예를 들어 일별 판매량, 시간별 전력 사용량, 주간 방문자 수처럼 시간 순서가 있는 데이터를 보고 앞으로의 값을 맞히는 문제를 떠올리면 됩니다.

AutoGluon은 단일 점 예측뿐 아니라 **확률적 예측(probabilistic forecasting)** 도 지원합니다. 즉, 미래를 하나의 숫자로만 찍는 것이 아니라 "대체로 이 정도일 것 같고, 낮으면 이 정도, 높으면 이 정도까지 흔들릴 수 있다"는 범위까지 함께 예측합니다.

- `prediction_length`: 앞으로 몇 시점을 예측할지 정하는 값입니다. 일별 데이터에서 `7`이면 7일, 시간별 데이터에서 `48`이면 48시간을 예측합니다.
- 평균 예측값(`mean`)뿐 아니라 분위수(`quantile`) 예측도 함께 만들 수 있습니다.
- 그래서 "얼마나 될까"뿐 아니라 "얼마나 불확실한가"도 함께 볼 수 있습니다.

예를 들어 예측 결과가 아래처럼 나왔다고 해보겠습니다.

- `mean = 100`
- `0.1 quantile = 80`
- `0.9 quantile = 130`

이 값은 보통 "중심적으로는 100 정도로 보이지만, 꽤 낮으면 80 근처, 꽤 높으면 130 근처까지 갈 수 있다" 정도로 읽습니다. 재고 계획이나 인력 배치처럼 불확실성이 중요한 문제에서는 이런 범위 정보가 특히 유용합니다.

아래 예제에서는 M4 Daily 데이터를 사용해 이후 개념들을 차례대로 붙여 봅니다.


In [2]:
df = pd.read_csv("https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_daily_subset/train.csv")
df.head()

,item_id,timestamp,target
0,D1737,1995-05-23,1900.0
1,D1737,1995-05-24,1877.0
2,D1737,1995-05-25,1873.0
3,D1737,1995-05-26,1859.0
4,D1737,1995-05-27,1876.0


## 3. Static Features 붙이기

`static features`는 시간에 따라 변하지 않는 메타데이터입니다. 쉽게 말해, 각 시계열을 설명하는 "배경 정보"입니다. 예를 들면 지역, 상품 카테고리, 브랜드, 매장 유형, 센서 종류 같은 정보가 여기에 해당합니다.

문서 예제의 M4 Daily 데이터에는 각 시계열이 어떤 도메인(`domain`)에서 왔는지 알려주는 정적 특성이 함께 제공됩니다. 예를 들어 어떤 시계열은 `Finance`, 어떤 시계열은 `Industry`, 어떤 시계열은 `Micro` 같은 그룹에 속합니다.

이런 정보가 중요한 이유는 서로 비슷한 성격의 시계열끼리 패턴을 공유할 수 있기 때문입니다. 예를 들어 같은 산업군에 속한 시계열은 계절성이나 변동성 구조가 비슷할 수 있습니다. 모델은 정적 특성을 이용해 "이 시계열은 어떤 부류인가"를 함께 참고할 수 있습니다.

참고로 AutoGluon은 정적 특성의 dtype을 보고 범주형/연속형을 자동으로 추론합니다.

- `object`, `string`, `category` -> 범주형
- `int`, `float` -> 연속형
- 학습 시 정적 특성을 사용했다면 이후 `predict()`, `evaluate()`, `leaderboard()`에 넣는 데이터도 같은 컬럼 이름과 dtype을 맞추는 것이 중요합니다.

정리하면, `target`이 시간에 따라 바뀌는 본문이라면 `static features`는 각 시계열의 프로필 정보라고 볼 수 있습니다.


In [3]:
static_features_df = pd.read_csv(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_daily_subset/metadata.csv"
)

train_data = TimeSeriesDataFrame.from_data_frame(
    df,
    id_column="item_id",
    timestamp_column="timestamp",
    static_features_df=static_features_df,
)

train_data.static_features.head()


,domain
item_id,
D1737,Industry
D1843,Industry
D2246,Finance
D909,Micro
D1345,Micro


## 4. Time-varying Covariates 추가하기

공변량(covariates)은 시간에 따라 변하는 추가 입력입니다. 즉, 예측 대상인 `target` 외에도 모델이 함께 볼 수 있는 보조 신호라고 생각하면 됩니다. 판매량을 예측할 때 요일, 휴일 여부, 프로모션 여부, 날씨 같은 정보를 같이 넣는 상황이 대표적입니다.  
AutoGluon 문서에서는 이를 두 가지로 구분합니다. 핵심 차이는 **예측 시점의 미래 값까지 알고 있느냐** 입니다. 예측할 때 모델이 사용할 수 있는 정보가 다르기 때문입니다.

- `known covariates`: 예측 구간 전체에 대해 미래 값까지 미리 알고 있는 변수입니다.
- 예: 요일, 월, 휴일, 가격 정책, 프로모션 계획
- `past covariates`: 예측 시작 시점까지만 알고 있는 변수입니다.
- 예: 다른 상품 판매량, 실제 기온, `target`의 변형값

예를 들어 앞으로 14일의 판매량을 예측한다고 하면, 다음 14일이 주말인지 아닌지는 달력을 보면 미리 알 수 있으므로 `known covariate`입니다. 반면 미래 14일의 실제 기온이나 미래 판매량의 로그값은 지금 시점에서는 알 수 없으므로 `past covariate` 쪽에 가깝습니다.

아래에서는 문서 흐름을 따라 다음 두 변수를 추가합니다.

- `log_target`: 과거에만 알 수 있는 `past covariate`
- `log_target`은 `target`의 과거값을 로그 스케일로 변환한 보조 feature입니다.
- 새로운 정보를 추가한다기보다는, 같은 값을 다른 스케일로 한 번 더 보여주는 역할에 가깝습니다.
- 값의 범위가 너무 크거나, 절대 차이보다 비율 변화가 더 중요할 때 패턴을 더 쉽게 보이게 만들 수 있습니다.
- 다만 모든 문제에서 항상 성능 향상을 보장하는 것은 아니며, 문서에서는 `past covariate` 예시로 이해하는 것이 좋습니다.
- `weekend`: 미래 일정상 미리 알 수 있는 `known covariate`
- 주말 여부는 미래 날짜만 알면 만들 수 있으므로 예측 구간에도 모델에 제공할 수 있습니다.

즉 이 예제에서 `target`은 우리가 맞히고 싶은 값이고, `log_target`과 `weekend`는 그 예측을 돕는 추가 힌트입니다.


In [4]:
train_data["log_target"] = np.log(train_data["target"])

WEEKEND_INDICES = [5, 6]
timestamps = train_data.index.get_level_values("timestamp")
train_data["weekend"] = timestamps.weekday.isin(WEEKEND_INDICES).astype(float)

train_data.head()


target  log_target  weekend
item_id timestamp                              
D1737   1995-05-23  1900.0    7.549609      0.0
        1995-05-24  1877.0    7.537430      0.0
        1995-05-25  1873.0    7.535297      0.0
        1995-05-26  1859.0    7.527794      0.0
        1995-05-27  1876.0    7.536897      1.0

## 5. 미래 구간의 Known Covariates 만들기

`TimeSeriesPredictor`를 만들 때 `known_covariates_names`에 컬럼 이름을 넣어 두면, AutoGluon은 그 컬럼을 예측 시점에도 제공받아야 하는 미래 정보로 해석합니다. 즉, 모델은 과거 학습 구간뿐 아니라 앞으로 예측할 구간에서도 이 값을 입력으로 받는다고 가정합니다.

중요한 점은 다음과 같습니다.

- `target`과 `known_covariates_names`를 제외한 나머지 열은 자동으로 `past covariates`로 취급됩니다.
- `known_covariates`에는 모든 `item_id`가 포함되어야 합니다.
- 각 시계열의 마지막 시점 이후 `prediction_length`만큼의 미래 timestamp가 포함되어야 합니다.

왜 이런 과정이 필요할까요? 예를 들어 `weekend`는 과거 데이터 안에도 있지만, 예측을 하려면 **미래 14일에 대해서도** 주말 여부를 모델에 알려줘야 합니다. 과거 데이터만으로는 미래 행이 없기 때문에, 예측용 미래 프레임을 별도로 만들어서 넣어야 합니다.

문서에서 소개하는 `make_future_data_frame()`은 이런 미래 프레임을 만들 때 가장 편한 출발점입니다. 이 함수는 각 시계열별로 필요한 미래 timestamp 뼈대를 자동으로 만들어 주고, 우리는 그 위에 `weekend` 같은 값을 채워 넣으면 됩니다.


In [5]:
prediction_length = 14

covariate_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    known_covariates_names=["weekend"],
    freq=train_data.freq,
    eval_metric="MASE",
)

known_covariates = covariate_predictor.make_future_data_frame(train_data)
known_covariates["weekend"] = (
    known_covariates["timestamp"].dt.weekday.isin(WEEKEND_INDICES).astype(float)
)

known_covariates.head()


,item_id,timestamp,weekend
0,D1069,2013-01-20,1.0
1,D1069,2013-01-21,0.0
2,D1069,2013-01-22,0.0
3,D1069,2013-01-23,0.0
4,D1069,2013-01-24,0.0


## 6. Holiday Feature 예시

휴일은 대표적인 `known covariate`입니다. 주말과 비슷하게, 미래 날짜의 공휴일 여부는 달력만 있으면 미리 알 수 있기 때문입니다.

원문 튜토리얼은 `holidays` 패키지로 국가별 공휴일을 생성하는 방법을 보여주지만, 여기서는 추가 의존성 없이 구조만 이해할 수 있도록 간단한 커스텀 딕셔너리로 예시를 만듭니다.

휴일 feature를 넣는 이유는 특정 날짜에 수요가 튀거나 꺼지는 패턴을 모델이 더 직접적으로 볼 수 있게 하기 위해서입니다. 예를 들어 블랙프라이데이, 설날, 추석, 크리스마스 전후에는 평소와 다른 패턴이 자주 나타납니다.

또한 휴일 feature는 한 가지 방식만 있는 것이 아닙니다.

- 특정 휴일별로 각각 one-hot 컬럼을 만들 수 있고
- 단순히 `Holiday` 여부만 0/1로 표시하는 컬럼 하나만 만들 수도 있습니다.

실무에서는 반드시 **예측 대상 국가/지역에 맞는 휴일 캘린더**를 사용해야 합니다. 잘못된 국가의 휴일을 넣으면 오히려 잡음을 추가할 수 있습니다.


In [6]:
custom_holidays = {
    datetime.date(1995, 1, 29): "Superbowl",
    datetime.date(1995, 11, 24): "Black Friday",
    datetime.date(1996, 1, 28): "Superbowl",
    datetime.date(1996, 11, 29): "Black Friday",
}


def add_holiday_features(
    ts_df: TimeSeriesDataFrame,
    holiday_dict: dict,
    include_individual_holidays: bool = True,
    include_holiday_indicator: bool = True,
) -> TimeSeriesDataFrame:
    ts_df = ts_df.copy()
    timestamps = ts_df.index.get_level_values("timestamp")
    holiday_dummies = pd.get_dummies(pd.Series(holiday_dict)).astype(float)
    holiday_frame = holiday_dummies.reindex(timestamps.date).fillna(0)

    if include_individual_holidays:
        ts_df[holiday_frame.columns] = holiday_frame.values
    if include_holiday_indicator:
        ts_df["Holiday"] = holiday_frame.max(axis=1).values

    return ts_df


train_data_with_holidays = add_holiday_features(train_data, custom_holidays)
holiday_columns = train_data_with_holidays.columns.difference(train_data.columns)

print("추가된 holiday 컬럼:", list(holiday_columns))
train_data_with_holidays.head()


추가된 holiday 컬럼: ['Black Friday', 'Holiday', 'Superbowl']


target  log_target  weekend  Black Friday  Superbowl  \
item_id timestamp                                                          
D1737   1995-05-23  1900.0    7.549609      0.0           0.0        0.0   
        1995-05-24  1877.0    7.537430      0.0           0.0        0.0   
        1995-05-25  1873.0    7.535297      0.0           0.0        0.0   
        1995-05-26  1859.0    7.527794      0.0           0.0        0.0   
        1995-05-27  1876.0    7.536897      1.0           0.0        0.0   

                    Holiday  
item_id timestamp            
D1737   1995-05-23      0.0  
        1995-05-24      0.0  
        1995-05-25      0.0  
        1995-05-26      0.0  
        1995-05-27      0.0

## 7. 데이터 형식, 길이 조건, 결측 처리

AutoGluon은 기본적으로 `TimeSeriesDataFrame` 형식을 기대합니다. 또한 내부 검증 세트를 만들 수 있을 만큼 **일부 시계열은 충분히 길어야** 합니다. 시계열이 너무 짧으면 학습에 쓸 과거 구간과 검증에 쓸 미래 구간을 동시에 확보하기 어렵기 때문입니다.

- 기본 설정에서는 적어도 몇 개의 시계열이 `max(prediction_length + 1, 5) + prediction_length` 이상 길어야 합니다.
- 이 식은 "예측할 미래 구간 `prediction_length`"와, 그 앞에서 모델이 참고할 최소 과거 길이를 함께 확보해야 한다는 뜻으로 읽으면 됩니다.
- `num_val_windows`, `val_step_size`를 늘리면 더 긴 시계열이 필요합니다.
- 이유는 내부 검증을 한 번이 아니라 여러 번, 서로 다른 cutoff에서 반복해야 하기 때문입니다. 즉 검증용 미래 구간을 여러 개 떼어내야 하므로 더 긴 데이터가 필요합니다.

또한 현실의 데이터는 항상 깔끔한 일간/시간간격 형태가 아닙니다. 금융 데이터처럼 시점이 불규칙하거나, 센서 로그처럼 중간 값이 비어 있는 경우가 흔합니다. 이때는 아래 두 가지 접근을 많이 씁니다.

- `freq`를 지정해서 예측기가 원하는 빈도를 명시합니다.
- `convert_frequency()`로 index를 정규화하고, 필요하면 `fill_missing_values()`로 NaN을 처리합니다.

아래 예제는 불규칙한 시계열을 일 단위로 맞춘 뒤, 결측을 채우는 두 가지 방법을 보여 줍니다.

- 기본 채우기: 주변 값 기준으로 결측을 메움
- 상수 채우기: 결측을 0 같은 고정값으로 채움

대부분의 모델은 NaN을 어느 정도 처리할 수 있지만, 수요 예측처럼 결측을 "관측 없음 = 0"으로 보는 도메인에서는 상수 채우기가 더 적절할 수 있습니다.


In [7]:
df_irregular = TimeSeriesDataFrame(
    pd.DataFrame(
        {
            "item_id": [0, 0, 0, 1, 1],
            "timestamp": [
                "2022-01-01",
                "2022-01-02",
                "2022-01-04",
                "2022-01-01",
                "2022-01-04",
            ],
            "target": [1, 2, 3, 4, 5],
        }
    )
)

df_regular = df_irregular.convert_frequency(freq="D")
df_filled = df_regular.fill_missing_values()
df_filled_zero = df_regular.fill_missing_values(method="constant", value=0.0)

print(f"원래 빈도: {df_irregular.freq}")
print(f"변환 후 빈도: {df_regular.freq}")

display(df_regular)
display(df_filled)
display(df_filled_zero)


원래 빈도: None
변환 후 빈도: D


target
item_id timestamp         
0       2022-01-01     1.0
        2022-01-02     2.0
        2022-01-03     NaN
        2022-01-04     3.0
1       2022-01-01     4.0
        2022-01-02     NaN
        2022-01-03     NaN
        2022-01-04     5.0

target
item_id timestamp         
0       2022-01-01     1.0
        2022-01-02     2.0
        2022-01-03     2.0
        2022-01-04     3.0
1       2022-01-01     4.0
        2022-01-02     4.0
        2022-01-03     4.0
        2022-01-04     5.0

target
item_id timestamp         
0       2022-01-01     1.0
        2022-01-02     2.0
        2022-01-03     0.0
        2022-01-04     3.0
1       2022-01-01     4.0
        2022-01-02     0.0
        2022-01-03     0.0
        2022-01-04     5.0

## 8. 평가용 데이터 분리와 `evaluate()`

예측 성능을 보려면 학습에 쓰지 않을 미래 구간을 남겨 둬야 합니다. 시계열 문제에서는 데이터를 랜덤하게 섞어 나누지 않고, 보통 **가장 뒤쪽의 미래 구간**을 따로 떼어내 평가합니다. 그래야 실제 예측 상황과 비슷해지기 때문입니다.

`train_test_split(prediction_length)`는 각 시계열의 마지막 `prediction_length` 구간을 떼어내는 가장 간단한 방법입니다.

- `train_data`: 마지막 `prediction_length`를 제외한 과거 구간
- `test_data`: 과거 + 미래를 모두 포함한 전체 구간

여기서 `test_data`가 전체 구간을 가지고 있다는 점이 중요합니다. 모델은 과거 구간을 입력으로 보고, 마지막 `prediction_length` 길이의 미래 구간을 맞히게 됩니다.

`evaluate()`는 `test_data`의 마지막 `prediction_length` 시점을 hold-out으로 사용해 점수를 계산합니다. 반환값은 기본적으로 **내부 검증에서 가장 좋았던 모델**의 점수입니다.

즉, `fit()` 때 AutoGluon 내부에서 한 번 모델 선택을 하고, `evaluate()`에서는 우리가 따로 남겨 둔 외부 평가 구간에서 그 모델이 실제로 어느 정도 성능을 내는지 확인하는 흐름입니다.


In [ ]:
prediction_length = 48

full_data = TimeSeriesDataFrame.from_path(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/electricity_small/test.csv"
)
train_eval_data, test_eval_data = full_data.train_test_split(prediction_length)

eval_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
    path="autogluon-timeseries-eval",
)

eval_predictor.fit(
    train_eval_data,
    hyperparameters={
        "SeasonalNaive": {},
        "RecursiveTabular": {},
        "Chronos": {},
    },
    verbosity=0,
)

eval_predictor.evaluate(test_eval_data)


## 9. `leaderboard()`와 예측 구간 시각화

`leaderboard()`를 쓰면 모든 모델의 `score_test`, `score_val`, 예측 시간, 학습 시간을 한 번에 비교할 수 있습니다. 즉, "어떤 모델이 가장 잘 맞았는가"뿐 아니라 "얼마나 빨리 학습하고 예측하는가"도 함께 볼 수 있습니다.

보통은 아래처럼 읽으면 됩니다.

- `score_val`: 학습 중 내부 검증 점수
- `score_test`: 우리가 따로 남겨 둔 테스트 구간 점수
- `fit_time_*`: 학습 시간
- `pred_time_*`: 예측 시간

아래 시각화는 `train_data`와 `test_data`의 차이를 보여 줍니다. 주황색으로 칠한 부분이 바로 모델이 맞혀야 하는 **forecast horizon** 입니다. 이 그림을 보면, 시계열 예측에서 테스트란 과거 전체를 버리는 것이 아니라 **마지막 미래 구간만 떼어내는 일**이라는 점이 더 직관적으로 보입니다.


In [ ]:
display(eval_predictor.leaderboard(test_eval_data))

item_id = test_eval_data.item_ids[0]
max_history_length = 300

train_ts = train_eval_data.loc[item_id]["target"].iloc[-max_history_length:]
test_ts = test_eval_data.loc[item_id]["target"].iloc[-(max_history_length + prediction_length):]

fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=[10, 4], sharex=True)
ax1.set_title("Train data (past time series values)")
ax1.plot(train_ts)
ax2.set_title("Test data (past + future time series values)")
ax2.plot(test_ts)

for ax in (ax1, ax2):
    ax.fill_between(
        np.array([train_ts.index[-1], test_ts.index[-1]]),
        float(test_ts.min()),
        float(test_ts.max()),
        color="C1",
        alpha=0.3,
        label="Forecast horizon",
    )

plt.legend()
plt.show()


## 10. 여러 cutoff로 Backtesting 하기

단일 hold-out 구간만으로는 운이 좋거나 나쁜 구간에 성능이 좌우될 수 있습니다. 예를 들어 마지막 1주일이 유난히 쉬운 구간이거나, 반대로 매우 특이한 구간일 수도 있습니다. 그래서 문서에서는 여러 시점에서 잘라 보는 **multi-window backtesting** 을 권장합니다.

- `cutoff=-prediction_length`가 기본 평가 위치입니다.
- 더 과거 위치까지 잘라 보려면 테스트 데이터를 더 길게 남겨 둬야 합니다.
- 창(window)을 늘리면 성능 추정은 더 안정적이지만, 학습에 쓰는 데이터는 줄어듭니다.

예를 들어 `prediction_length=24`이고 backtest window를 3개 사용한다면, 마지막 24시점만 보는 것이 아니라 그보다 앞쪽 구간까지 포함해 총 3번 평가하게 됩니다. 이렇게 하면 특정 구간에 성능이 과하게 의존하는지 더 잘 볼 수 있습니다.


In [ ]:
num_test_windows = 3
train_multi, test_multi = full_data.train_test_split(num_test_windows * prediction_length)

backtest_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
    path="autogluon-timeseries-backtest",
)

backtest_predictor.fit(
    train_multi,
    hyperparameters={
        "SeasonalNaive": {},
        "RecursiveTabular": {},
        "Chronos": {},
    },
    verbosity=0,
)

leaderboards = []
for cutoff in range(-num_test_windows * prediction_length, 0, prediction_length):
    lb = backtest_predictor.leaderboard(test_multi, cutoff=cutoff)
    lb["cutoff"] = cutoff
    leaderboards.append(lb[["model", "score_test", "cutoff"]])

scores_per_window = pd.concat(leaderboards).pivot(
    index="model", columns="cutoff", values="score_test"
)
scores_per_window


## 11. Backtest Predictions 시각화

`backtest_predictions()`와 `backtest_targets()`를 사용하면 각 backtest window별 예측 결과를 직접 들여다볼 수 있습니다. 단순히 평균 점수 하나만 보는 대신, 모델이 어느 구간에서 잘 맞고 어느 구간에서 흔들리는지 확인할 때 특히 유용합니다.

예를 들어 어떤 모델이 평균 점수는 좋아 보여도, 특정 이벤트 구간에서는 계속 크게 빗나갈 수 있습니다. 이런 문제는 표 한 장만 보고는 놓치기 쉽고, backtest 예측을 직접 그려 보면 훨씬 잘 드러납니다.


In [ ]:
predictions_per_window = backtest_predictor.backtest_predictions(
    test_multi, num_val_windows=num_test_windows
)
targets_per_window = backtest_predictor.backtest_targets(
    test_multi, num_val_windows=num_test_windows
)

item_ids = test_multi.item_ids[:4]
all_predictions = pd.concat(predictions_per_window)

backtest_predictor.plot(
    test_multi,
    all_predictions,
    max_history_length=300,
    item_ids=item_ids,
)

for cutoff in range(-num_test_windows * prediction_length, 0, prediction_length):
    for i, ax in enumerate(plt.gcf().axes):
        cutoff_timestamp = test_multi.loc[item_ids[i]].index[cutoff]
        ax.axvline(cutoff_timestamp, color="gray", linestyle="--", alpha=0.7)

plt.show()


## 12. AutoGluon 시계열 모델 종류

문서에서는 AutoGluon의 예측 모델을 크게 세 부류로 나눕니다. 어떤 모델이 더 낫다고 단정하기보다는, 데이터의 수, 시계열 간 유사성, 공변량 유무, 학습 시간 제약에 따라 강점이 달라진다고 이해하는 편이 좋습니다.

- `local models`: 각 시계열마다 별도 통계 모델을 학습합니다.
- 예: `ETS`, `AutoARIMA`, `Theta`, `SeasonalNaive`
- 장점: 해석이 비교적 쉽고, 개별 시계열 패턴을 직접 따라가기 좋습니다.
- `global models`: 여러 시계열을 한 번에 보고 하나의 모델을 학습합니다.
- 예: `DeepAR`, `PatchTST`, `DLinear`, `TemporalFusionTransformer`, `Chronos`
- 여기에 `RecursiveTabular`, `DirectTabular` 같은 tabular 기반 글로벌 모델도 포함할 수 있습니다.
- 장점: 시계열이 많고 서로 공통 패턴이 있을 때 강하며, covariates 활용이 좋은 경우가 많습니다.
- `ensemble models`: 여러 모델 예측을 결합합니다.
- 기본적으로 `WeightedEnsemble`이 자동으로 붙습니다.
- 장점: 서로 다른 모델의 강점을 평균적으로 묶어 더 안정적인 성능을 기대할 수 있습니다.

실전에서는 보통 한 가지 모델만 고집하기보다, 빠른 베이스라인 모델과 강한 글로벌 모델을 함께 돌려 보고 결과를 비교하는 식으로 접근합니다. 또한 모든 모델이 정적 특성, known covariates, past covariates를 동일하게 지원하는 것은 아니므로, 자세한 지원 여부는 Model Zoo 문서를 함께 보는 것이 좋습니다.


## 13. `TimeSeriesPredictor` 고급 설정

`TimeSeriesPredictor.fit()`은 초보자용 preset부터 고급 사용자를 위한 수동 설정, HPO까지 폭넓게 지원합니다. 처음에는 preset으로 시작하고, 필요할 때만 세부 설정을 늘려 가는 방식이 가장 실용적입니다.

- `presets`: 빠른 실험과 품질 수준을 한 번에 정합니다.
- 예: `fast_training`, `medium_quality`, `high_quality`, `best_quality`
- `time_limit`: 전체 학습 시간을 초 단위로 제한합니다.
- `hyperparameters`: 학습할 모델과 설정을 직접 지정합니다.
- `hyperparameter_tune_kwargs`: HPO trial 수와 검색 방식을 지정합니다.
- `num_val_windows`: 내부 검증 창 수를 늘려 모델 선택을 더 안정적으로 할 수 있습니다.

각 옵션은 아래처럼 이해하면 됩니다.

- 빠르게 감을 보고 싶다 -> `presets` + `time_limit`
- 특정 모델만 골라서 실험하고 싶다 -> `hyperparameters`
- 모델 내부 설정까지 탐색하고 싶다 -> `hyperparameter_tune_kwargs`
- 검증을 더 엄격하게 하고 싶다 -> `num_val_windows`

아래 셀은 문서에 나온 대표 설정 패턴만 한 번에 모아 둔 참고용 예시입니다. 실제로는 한 번에 모든 옵션을 건드리기보다, baseline을 만든 뒤 하나씩 늘려 가는 것이 디버깅과 비교에 훨씬 유리합니다.


In [ ]:
from autogluon.common import space

# 1) preset + time_limit
preset_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
)
# preset_predictor.fit(train_eval_data, presets="medium_quality", time_limit=60 * 30)

# 2) 특정 모델만 직접 지정
manual_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
)
# manual_predictor.fit(
#     train_eval_data,
#     hyperparameters={
#         "DeepAR": {},
#         "Theta": [
#             {"decomposition_type": "additive"},
#             {"seasonal_period": 1},
#         ],
#     },
# )

# 3) HPO
hpo_predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    eval_metric="MASE",
)
# hpo_predictor.fit(
#     train_eval_data,
#     hyperparameters={
#         "DeepAR": {
#             "hidden_size": space.Int(20, 100),
#             "dropout_rate": space.Categorical(0.1, 0.3),
#         },
#     },
#     hyperparameter_tune_kwargs={
#         "num_trials": 20,
#         "searcher": "random",
#         "scheduler": "local",
#     },
#     enable_ensemble=False,
# )

# 4) 내부 검증 창 수 늘리기
# preset_predictor.fit(train_eval_data, num_val_windows=3)

# 5) 사용자가 직접 validation set 제공
# preset_predictor.fit(train_data=train_eval_data, tuning_data=my_validation_dataset)


## 정리

이 노트북에서 정리한 핵심은 다음과 같습니다.

1. AutoGluon은 평균값뿐 아니라 분위수까지 포함한 **확률적 예측**을 수행할 수 있습니다.
2. 정적 특성(`static features`)과 시점별 공변량(`known/past covariates`)을 함께 넣어 더 현실적인 예측 문제를 다룰 수 있습니다.
3. irregular data는 `convert_frequency()`와 `fill_missing_values()`로 정리할 수 있습니다.
4. `evaluate()`, `leaderboard()`, backtesting으로 예측 품질을 더 신뢰성 있게 점검할 수 있습니다.
5. `presets`, 수동 `hyperparameters`, HPO를 통해 학습 전략을 문제 상황에 맞게 조절할 수 있습니다.

Quick Start에서 전체 흐름을 먼저 익혔다면, 이 노트북은 실전에서 자주 마주치는 "추가 정보 활용", "평가 설계", "모델 제어"를 연결해 보는 용도로 쓰면 좋습니다.

특히 처음에는 아래 세 가지만 확실히 구분해도 이해가 훨씬 쉬워집니다.

- `prediction_length`: 앞으로 몇 칸을 예측할지
- `known covariates`: 미래 값까지 미리 알고 있는 보조 입력
- `past covariates`: 과거까지만 아는 보조 입력

이 세 개가 익숙해지면 이후의 static feature, holiday feature, backtesting, 모델 선택도 훨씬 자연스럽게 이어집니다.
